# Task Execution: With and Without Skills

This notebook compares two runs of the same word-frequency task to highlight the concrete benefits of skill-augmented execution:

- **Accuracy (Option B):** Without a skill the LLM must count words by reasoning alone — a task it reliably gets wrong. With the `word-frequency` skill a deterministic Python script is executed, producing exact counts every time.
- **Efficiency (Option A):** The skill gives the agent a precise action plan, so it reaches the goal in fewer steps than when it has to figure out the approach itself.

The comparison is intentionally simple: the same passage, the same agent, the same LLM — the only variable is whether the `word-frequency` skill is available.

## Setup Instructions

To ensure you have the required dependencies to run this notebook, you'll need to have our `llm-agents-from-scratch` framework installed on the running Jupyter kernel. To do this, you can launch this notebook with the following command while within the project's root directory:

```sh
uv run --with jupyter jupyter lab
```

Alternatively, if you just want to use the published version of `llm-agents-from-scratch` without local development, you can install it from PyPI by uncommenting the cell below.

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama installed on your local machine and have its LLM hosting service running. To download Ollama, follow the instructions found on this page: https://ollama.com/download. After downloading and installing Ollama, you can start a service by opening a terminal and running the command `ollama serve`.

## Setup

In [ ]:
import logging

from llm_agents_from_scratch import TOOLS_FOR_SKILL_RESOURCES, LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.logger import enable_console_logging

enable_console_logging(logging.INFO)

llm = OllamaLLM(model="qwen3:14b", think=False)

PASSAGE = """\
Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex.
Complex is better than complicated.
Flat is better than nested.
Sparse is better than dense.
Readability counts.
Special cases aren't special enough to break the rules.
Although practicality beats purity.
Errors should never pass silently.
Unless explicitly silenced.
In the face of ambiguity, refuse the temptation to guess.
There should be one-- and preferably only one --obvious way to do it.
Although that way may not be obvious at first unless you're Dutch.
Now is better than never.
Although never is often better than right now.
If the implementation is hard to explain, it's a bad idea.
If the implementation is easy to explain, it may be a good idea.
Namespaces are one honking great idea -- let's do more of those!
"""

## Without Skill

The agent is given the task with `skills_scopes=[]` so no skills are discovered at all. It must count word frequencies by reasoning alone — no script, no tool, just the LLM.

In [ ]:
task = Task(
    instruction=(
        "What are the top-5 most frequent words in the following"
        f" passage?\n\n{PASSAGE}"
    ),
)
agent = LLMAgent(llm=llm)
handler_without = agent.run(task, skills_scopes=[])

In [ ]:
await handler_without
result_without = handler_without.exception() or handler_without.result()
print(result_without)

## With Skill

The same agent is now given `TOOLS_FOR_SKILL_RESOURCES` and explicitly activates the `word-frequency` skill. The skill directs it to pipe the passage to `scripts/word_freq.py` via stdin — the Python script produces exact, deterministic counts.

In [ ]:
agent_with_skill = LLMAgent(llm=llm, tools=TOOLS_FOR_SKILL_RESOURCES)
handler_with = agent_with_skill.run_with_skill(
    "word-frequency",
    prompt=PASSAGE,
)

In [ ]:
await handler_with
result_with = handler_with.exception() or handler_with.result()
print(result_with)

## Comparison

`handler.step_counter` records how many `TaskStep`s were executed. We also verify accuracy against the ground-truth counts produced by running `word_freq.py` directly.

In [ ]:
print("| Metric            | Without skill | With skill |")
print("|-------------------|---------------|------------|")
print(
    f"| Steps taken       |"
    f" {handler_without.step_counter:<13} |"
    f" {handler_with.step_counter:<10} |",
)
print(
    f"| Accurate counts?  | {'no':<13} | {'yes':<10} |",
)